In [ ]:
import pickle
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [21]:
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3015.99it/s]


In [22]:
dataset = pd.read_csv('../datasets/SQuAD_processed.csv')

In [23]:
dataset.sample(5)

,context,question,label,jaccard,context_length,query_coverage,shared_tokens
79133,The show had originally planned on having four...,What radio DJ was originally hired as a judge ...,1,0.125000,685,0.785714,11
99481,"Unincorporated communities, localities and pla...",Who was Schwarzenegger's closest rival in the ...,0,0.033333,162,0.090909,1
43263,Digital-RGB LEDs are RGB LEDs that contain the...,What makes RGB LEDs different?,1,0.026316,618,0.400000,2
54307,Orthodox Judaism collectively considers itself...,What do Orthodox Jewish movements consider all...,0,0.094595,725,0.700000,7
62676,High-power LEDs (HP-LEDs) or high-output LEDs ...,What can one high-power LED replace?,1,0.052632,565,0.666667,4


In [24]:
q_emb = embedding_model.encode(dataset['question'].tolist())
c_emb = embedding_model.encode(dataset['context'].tolist())

In [25]:
def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1)
    vec2 = np.asarray(vec2)

    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return np.dot(vec1, vec2)/(norm1 * norm2)

cos_sims = np.array([cosine_similarity(q,c) for q,c in zip(q_emb,c_emb)]).reshape(-1,1)

In [26]:
X = np.hstack([q_emb,c_emb,cos_sims,dataset[['jaccard','context_length','query_coverage','shared_tokens']].values])
Y = dataset['label']

In [27]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=42,stratify=Y)

In [28]:
model = XGBClassifier()

In [29]:
model.fit(X_train,Y_train) 

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [45]:
y_p = model.predict(X_train) 
acc = accuracy_score(Y_train, y_p)
print(f"Training Accuracy: {acc}")

Training Accuracy: 0.9740334019675132


In [46]:
y_p = model.predict(X_test) 
acc = accuracy_score(Y_test, y_p)
print(f"Validation Accuracy: {acc}")

Validation Accuracy: 0.9352963341766701


In [47]:
print(classification_report(Y_test, y_p))

              precision    recall  f1-score   support

           0       0.95      0.92      0.94     19190
           1       0.92      0.95      0.93     17964

    accuracy                           0.94     37154
   macro avg       0.94      0.94      0.94     37154
weighted avg       0.94      0.94      0.94     37154



In [44]:
with open("../models/sklearn_94.pkl", "wb") as f:
    pickle.dump(model, f)